# Conservation: the pendulum that went over the top

**Book:** §5.8, Figure 5.7 &nbsp;·&nbsp; `ch05/conservation_pendulum.ipynb`

$$\ddot\theta + \sin\theta = 0,\qquad \theta(0)=1.0,\ \dot\theta(0)=0$$

Both initial conditions are **hard**: $\theta = \theta_0 + t^2\,\mathcal N(t)$. The loss is the pure
residual. And the system conserves

$$E = \tfrac12\dot\theta^2 + (1-\cos\theta) = 0.4597,$$

which is **never imposed** — it is a free, independent check.

**What happens is not a gentle drift.** Over $t\in[0,6]$ the plain network is excellent (rel $L_2$
≈ 2e-4, energy drift 7e-4). Push to $t\in[0,10]$ and it does something far more interesting: it
**gains energy, climbs to $E=2$ — the separatrix — and goes over the top.** It converges to a
*rotating* pendulum instead of a swinging one. That is still a solution of the ODE. It is just not
*our* solution, and the residual is perfectly happy about it.

**The energy trace is what catches this**, and you can compute it without knowing the answer.

**Remedy implemented:** add $(E(t)-E_0)^2$ to the loss. It pins the trajectory to the correct
energy shell.

In [ ]:
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def g1(f, x): return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]
def mlp(s, seed=0):
    torch.manual_seed(seed); L = []
    for i in range(len(s)-1):
        L.append(nn.Linear(s[i], s[i+1]))
        if i < len(s)-2: L.append(nn.Tanh())
    return nn.Sequential(*L)
rel = lambda p, e: float(np.sqrt(np.mean((p-e)**2)/np.mean((e-e.mean())**2 + 1e-30)))
M = {}

In [ ]:
TH0, T8 = 1.0, 10.0                       # ~1.6 periods; large-amplitude (nonlinear) swing
E0 = 0.5*0.0**2 + (1 - np.cos(TH0))
ref = solve_ivp(lambda t,z:[z[1], -np.sin(z[0])], [0, 1.4*T8], [TH0, 0.0],
                rtol=1e-11, atol=1e-13, dense_output=True)
tg  = np.linspace(0, T8, 600);        th_ex  = ref.sol(tg)[0]
tex = np.linspace(0, 1.4*T8, 800);    th_exx = ref.sol(tex)[0]

def pendulum(w_inv, seed=0, epochs=20000):
    net = mlp([1,64,64,64,1], seed)
    TH = lambda t: TH0 + t*t*net(t/T8)                # theta(0)=TH0 and theta'(0)=0, both hard
    opt = torch.optim.Adam(net.parameters(), 3e-3)
    for e in range(epochs):
        if e == int(.6*epochs):
            for g in opt.param_groups: g['lr'] = 5e-4
        if e == int(.85*epochs):
            for g in opt.param_groups: g['lr'] = 1e-4
        opt.zero_grad()
        t = (torch.rand(512,1)*T8).requires_grad_(True)
        th = TH(t); thd = g1(th, t); thdd = g1(thd, t)
        loss = ((thdd + torch.sin(th))**2).mean()
        if w_inv > 0:                                  # Remedy (ii): penalise the invariant
            E = 0.5*thd**2 + (1 - torch.cos(th))
            loss = loss + w_inv*((E - E0)**2).mean()
        loss.backward(); opt.step()
    return TH

def evalp(TH, t):
    tt = torch.tensor(t, dtype=torch.float32).reshape(-1,1).requires_grad_(True)
    th = TH(tt); thd = g1(th, tt)
    th = th.detach().numpy().ravel(); thd = thd.detach().numpy().ravel()
    return th, 0.5*thd**2 + (1 - np.cos(th))

# Control: over a SHORTER horizon the plain network is excellent and the energy barely moves.
# The pathology is not a slow creeping drift -- it appears suddenly as the horizon lengthens.
T8_SHORT = 6.0
_T8 = T8; T8 = T8_SHORT
TH_short = pendulum(0.0)
ts = np.linspace(0, T8_SHORT, 400); th_s, E_s = evalp(TH_short, ts)
th_sref = ref.sol(ts)[0]
M['pend_short']  = rel(th_s, th_sref)
M['drift_short'] = float(np.abs(E_s-E0).max()/E0)
print(f'[5.8] SHORT horizon t<={T8_SHORT}: rel L2 = {M["pend_short"]:.2e}, drift = {M["drift_short"]:.1e}')
T8 = _T8

TH_plain = pendulum(0.0)
TH_cons  = pendulum(1.0)
th_p, E_p = evalp(TH_plain, tg)
th_c, E_c = evalp(TH_cons,  tg)
M['pend_plain']  = rel(th_p, th_ex);  M['pend_cons'] = rel(th_c, th_ex)
M['drift_plain'] = float(np.abs(E_p-E0).max()/E0)
M['drift_cons']  = float(np.abs(E_c-E0).max()/E0)
th_px, _ = evalp(TH_plain, tex)
M['extrap'] = rel(th_px[tex > T8], th_exx[tex > T8])
print(f'[5.8] plain      : rel L2 = {M["pend_plain"]:.2e},  energy drift = {M["drift_plain"]:.1e}')
print(f'[5.8] +invariant : rel L2 = {M["pend_cons"]:.2e},  energy drift = {M["drift_cons"]:.1e}')
print(f'[5.8] extrapolation beyond the training window: rel L2 = {M["extrap"]:.2f}')

fig, ax = plt.subplots(1, 3, figsize=(14.5, 4.2))
ax[0].plot(tg, th_ex, 'g', lw=2.8, alpha=.6, label='reference (RK)')
ax[0].plot(tg, th_p, 'r--', lw=1.5, label=f'plain PINN ({M["pend_plain"]:.1e})')
ax[0].plot(tg, th_c, 'b:', lw=1.8, label=f'+ invariant penalty ({M["pend_cons"]:.1e})')
ax[0].set_xlabel('t'); ax[0].set_ylabel(r'$\theta$'); ax[0].grid(alpha=.3); ax[0].legend(fontsize=8)
ax[0].set_title('(a) The plain PINN goes OVER THE TOP:\nit finds a rotation, not a swing', fontsize=10.5)
ax[1].axhline(E0, color='g', lw=2.8, alpha=.6, label='exact $E_0$')
ax[1].plot(tg, E_p, 'r--', lw=1.5, label=f'plain (drift {M["drift_plain"]:.1e})')
ax[1].plot(tg, E_c, 'b:', lw=1.8, label=f'penalised (drift {M["drift_cons"]:.1e})')
ax[1].set_xlabel('t'); ax[1].set_ylabel('energy $E(t)$'); ax[1].grid(alpha=.3); ax[1].legend(fontsize=8)
ax[1].axhline(2.0, color='k', ls='--', lw=1, alpha=.6)
ax[1].text(0.3, 2.03, 'separatrix, $E=2$', fontsize=8)
ax[1].set_title('(b) Energy climbs to the separatrix.\nThe residual never complains', fontsize=10.5)
ax[2].plot(tex, th_exx, 'g', lw=2.8, alpha=.6, label='reference')
ax[2].plot(tex, th_px, 'r--', lw=1.5, label='plain PINN')
ax[2].axvspan(T8, tex[-1], color='k', alpha=.08)
ax[2].axvline(T8, color='k', ls=':', lw=1.2)
ax[2].text(T8+0.3, 1.15, f'never sampled\nrel $L_2$ = {M["extrap"]:.2f}', fontsize=8.5)
ax[2].set_xlabel('t'); ax[2].set_ylabel(r'$\theta$'); ax[2].grid(alpha=.3); ax[2].legend(fontsize=8)
ax[2].set_title('(c) Outside the collocation window,\nthe network is meaningless', fontsize=10.5)
plt.tight_layout(); plt.show()